In [ ]:
import torch
import torch.nn.functional as F
import requests
import random
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
words = requests.get(url).text.splitlines()
print(words.__len__())
print(len(words))
if device := torch.device("mps" if torch.backends.mps.is_available() else "cpu"):
    print(f"Using device: {device}")


random.shuffle(words)
print(words[:10])
len(words)

In [ ]:
block_size = 7
epochs = 100
lr = 0.001

In [ ]:
chars = ['.'] + sorted(set(''.join(words)))
def stoi(s):
    return {c: i for i, c in enumerate(chars)}[s]
def itos(i):
    return {i: c for i, c in enumerate(chars)}[i]
print(itos(1), itos(2), itos(3), itos(0))
print(stoi('.'), stoi('a'), stoi('b'), stoi('c'))

In [ ]:
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi(ch)
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [ ]:
X, Y = build_dataset(words)
n = X.shape[0]
Xtr, Ytr = X[:int(n*0.9)], Y[:int(n*0.9)]
Xdev, Ydev = X[int(n*0.9):], Y[int(n*0.9):]
Xtr, Ytr = [x.to(device) for x in (Xtr, Ytr)]
Xdev, Ydev = [x.to(device) for x in (Xdev, Ydev)]
print(X.shape, Y.shape)
print(Xtr.shape, Ytr.shape)
print(Xdev.shape, Ydev.shape)

In [ ]:
train_ds = torch.utils.data.TensorDataset(Xtr.cpu(), Ytr.cpu())

train_dl = DataLoader(
    dataset = train_ds,
    batch_size = 1024,
    num_workers = 2,
    prefetch_factor = 2,
    pin_memory = False,
    persistent_workers = True,
)

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.C = torch.nn.Parameter(torch.randn((27, 10)))
        self.W1 = torch.nn.Parameter(torch.randn((70, 200)))
        self.b1 = torch.nn.Parameter(torch.randn(200))
        self.W2 = torch.nn.Parameter(torch.randn((200, 27)))
        self.b2 = torch.nn.Parameter(torch.randn(27))

    def forward(self, Xb):
        emb = self.C[Xb]
        emb = emb.view(-1, 70)
        h = torch.tanh(emb @ self.W1 + self.b1)
        logits = h @ self.W2 + self.b2
        return logits


In [ ]:
model = MLP().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9998)



In [ ]:
for e in range(epochs):
    for Xb, Yb in train_dl:
        Xb, Yb = Xb.to(device, non_blocking=True), Yb.to(device, non_blocking=True)
        logits = model(Xb)
        loss = F.cross_entropy(logits, Yb)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()
    if e % 10 == 0:
        print(loss.item(), scheduler.get_last_lr()[0])


In [ ]:
ix = torch.randint(0, Xdev.shape[0], (2000,), device=device)
Xb, Yb = Xdev[ix], Ydev[ix]
logits = model(Xb)
loss = F.cross_entropy(logits, Yb)
print(loss.item())

In [ ]:
for i in range(10):
    out = []
    context = [0] * block_size
    while True:
        logits = model(torch.tensor([context]))
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:] + [ix]
        out.append(itos(ix))
        if ix == 0:
            break
    print(''.join(out))